# 05 · Experimentos del recomendador por preferencias

**Proyecto:** Spotify Music Intelligence
**Módulo 5:** Recomendador por preferencias
**Objetivo:** validar presets editables, distancia ponderada (§15.6), detección
de perfiles fuera de distribución (§15.8) y diversidad MMR opcional (§15.9).

Unidad de modelado: `recording_group_id`. Se excluyen las grabaciones con
análisis acústico incompleto.

Las puntuaciones de similitud derivan de una distancia ponderada y **no son
probabilidades**.

## 1. Carga de configuración y presets

Los presets viven en `configs/presets.yaml` (AGENTS.md §32.6) y son editables
sin tocar Python. Cada preset define valores y pesos por variable (escala 0–3).

In [1]:
import os
from pathlib import Path

from spotify_intelligence.features.presets import load_presets, preset_names

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

presets = load_presets()
print("Presets:", preset_names())
print("Total presets:", len(presets))
print()
for key, preset in presets.items():
    print(f"{key}: {preset['label']} | pesos: {preset['weights']}")

Presets: ['entrenamiento_intenso', 'fiesta', 'concentracion_instrumental', 'relajacion', 'alegre_y_bailable', 'melancolico', 'acustico']
Total presets: 7

entrenamiento_intenso: Entrenamiento intenso | pesos: {'energy': 3, 'danceability': 2, 'valence': 1, 'acousticness': 2, 'instrumentalness': 1, 'tempo': 2}
fiesta: Fiesta | pesos: {'energy': 2, 'danceability': 3, 'valence': 2, 'acousticness': 1, 'instrumentalness': 1, 'tempo': 2}
concentracion_instrumental: Concentración instrumental | pesos: {'energy': 1, 'danceability': 1, 'valence': 1, 'acousticness': 2, 'instrumentalness': 3, 'tempo': 1}
relajacion: Relajación | pesos: {'energy': 2, 'danceability': 1, 'valence': 1, 'acousticness': 2, 'instrumentalness': 1, 'tempo': 2}
alegre_y_bailable: Alegre y bailable | pesos: {'energy': 2, 'danceability': 3, 'valence': 3, 'acousticness': 1, 'instrumentalness': 1, 'tempo': 2}
melancolico: Melancólico | pesos: {'energy': 2, 'danceability': 1, 'valence': 3, 'acousticness': 2, 'instrumentalness': 

## 2. Construcción de artefactos

Se construyen los artefactos versionados en `models/preferences/v1/`
(escalador, matriz escalada, catálogo e referencia OOD).

In [2]:
import json

from scripts.build_preference_recommender import build

manifest = build()
print(json.dumps(manifest, indent=2))

{
  "version": "v1",
  "pipeline": "preference_recommender",
  "features": [
    "energy",
    "danceability",
    "valence",
    "acousticness",
    "instrumentalness",
    "tempo"
  ],
  "scaler": "standard",
  "distance": "weighted_euclidean",
  "weight_scale": {
    "min": 0,
    "max": 3
  },
  "out_of_distribution": {
    "warning_percentile": 95,
    "weak_match_percentile": 99
  },
  "diversity": {
    "enabled_default": false,
    "method": "mmr",
    "lambda_default": 0.85
  },
  "filters": {},
  "presets_config": "configs/presets.yaml",
  "catalog_size": 83736,
  "dataset_sha256": "b202fa49909b2d5cef71a04b1d21243cfeb36414535f2ca9272aa646721177bd",
  "config_sha256": "1b68a0b7805b28133f7a4d2306edfe3c6994ff8c23e7049d6a0aa6f96013a506",
  "git_commit": "931394e3180f7f64ee19c09c5b7c81f67e0afada",
  "generated_at_utc": "2026-08-02T16:55:04.047297+00:00",
  "artifact_dir": "models\\preferences\\v1"
}


## 3. Recomendación por preset

Se recomienda con el preset «Fiesta» y se muestra el Top-5 ordenado por
distancia ponderada ascendente (mayor similitud primero).

In [3]:
from spotify_intelligence.recommenders.preference_based import (
    PreferenceProfile,
    PreferenceRecommender,
)

recommender = PreferenceRecommender("models/preferences/v1")

profile = PreferenceProfile.from_preset("fiesta")
print("Perfil:", profile.label, profile.values)

results = recommender.recommend(profile, top_n=5)
print(results[["track_name", "artists", "distance", "similarity"]].to_string(index=False))

Perfil: Fiesta {'energy': 0.85, 'danceability': 0.9, 'valence': 0.75, 'acousticness': 0.05, 'instrumentalness': 0.02, 'tempo': 128.0}


                    track_name                                artists  distance  similarity
             Big Apple - Mixed                              Plump DJs  0.038753    0.962693
                   Dirty Freak                         Mihalis Safras  0.062535    0.941146
      Mega Funk Reload Vuk Vuk                       DJ Ghost Floripa  0.090629    0.916902
        Elixir - Shade K Remix Shade k;Vazteria X;Zona Breakbeat DJ's  0.104912    0.905050
Give It To Me - Full Vocal Mix                           Matt Sassari  0.106900    0.903424


## 4. Perfil manual y rechazo de pesos cero

Se valida que un perfil con todos los pesos en cero es rechazado (§15.5).

In [4]:
from spotify_intelligence.recommenders.errors import InvalidPreferenceProfileError
from spotify_intelligence.recommenders.preference_based import PreferenceProfile

try:
    PreferenceProfile.from_manual(
        values={"energy": 0.8, "danceability": 0.9},
        weights={"energy": 0, "danceability": 0},
    )
except InvalidPreferenceProfileError as exc:
    print("Rechazado como se esperaba:", exc)

manual = PreferenceProfile.from_manual(
    values={
        "energy": 0.8,
        "danceability": 0.9,
        "valence": 0.5,
        "acousticness": 0.2,
        "instrumentalness": 0.1,
        "tempo": 120,
    },
    weights={
        "energy": 2,
        "danceability": 3,
        "valence": 1,
        "acousticness": 0,
        "instrumentalness": 0,
        "tempo": 2,
    },
)
print("Peso 0 ignora la variable; perfil manual válido.")

Rechazado como se esperaba: All weights are zero; the profile is not usable
Peso 0 ignora la variable; perfil manual válido.


## 5. Perfiles fuera de distribución (OOD)

Se compara la distancia del perfil al centroide del catálogo con los
percentiles p95/p99 precomputados (§15.8).

In [5]:
for key in ["fiesta", "melancolico", "concentracion_instrumental"]:
    profile = PreferenceProfile.from_preset(key)
    status = recommender.out_of_distribution_status(profile)
    print(
        f"{key}: {status['status']} | distancia centroide={status['distance_to_centroid']:.3f} "
        f"| p95={status['p95']:.3f} | p99={status['p99']:.3f}"
    )

fiesta: ok | distancia centroide=2.556 | p95=3.854 | p99=4.652
melancolico: ok | distancia centroide=2.612 | p95=3.854 | p99=4.652
concentracion_instrumental: ok | distancia centroide=3.034 | p95=3.854 | p99=4.652


## 6. Diversidad MMR opcional

Se compara el ranking puro contra el reranking MMR (`lambda = 0,85`). El primer
resultado coincide; las posiciones posteriores pueden reordenarse para aumentar
la diversidad interna.

In [6]:
profile = PreferenceProfile.from_preset("fiesta")
pure = recommender.recommend(profile, top_n=10)
mmr = recommender.recommend(profile, top_n=10, diversity_enabled=True, lambda_=0.85)

print(
    "Primer resultado idéntico:",
    pure.iloc[0]["recording_group_id"] == mmr.iloc[0]["recording_group_id"],
)
print("Similitud media pura:", round(pure["similarity"].mean(), 4))
print("Similitud media MMR: ", round(mmr["similarity"].mean(), 4))
print("Desviación interna pura:", round(pure["similarity"].std(), 4))
print("Desviación interna MMR: ", round(mmr["similarity"].std(), 4))

Primer resultado idéntico: True
Similitud media pura: 0.9074
Similitud media MMR:  0.9074
Desviación interna pura: 0.0265
Desviación interna MMR:  0.0265
